In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
tf.random.set_seed(42)
np.random.seed(42)

In [7]:
df = pd.read_csv("../data/Shark Tank US.csv")

In [8]:
y = df["Got Deal"].astype(int)

In [9]:
num_cols = ["Original Ask Amount", "Original Offered Equity",
            "Valuation Requested", "US Viewership", "Multiple Entrepreneurs"]
cat_cols = ["Industry", "Pitchers Gender"]
X = df[num_cols + cat_cols].copy()

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
])

In [12]:
X_train_p = preprocess.fit_transform(X_train).astype("float32")
X_test_p = preprocess.transform(X_test).astype("float32")

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_p.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

In [15]:
early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

In [16]:
history = model.fit(
    X_train_p, y_train.values,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[early],
    verbose=0,
)

In [17]:
proba = model.predict(X_test_p, verbose=0).ravel()
pred = (proba >= 0.5).astype(int)

In [20]:
print(round(roc_auc_score(y_test, proba), 4))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

0.5936
[[ 19  83]
 [ 18 135]]
              precision    recall  f1-score   support

           0      0.514     0.186     0.273       102
           1      0.619     0.882     0.728       153

    accuracy                          0.604       255
   macro avg      0.566     0.534     0.501       255
weighted avg      0.577     0.604     0.546       255



In [21]:
print(round(max(y_test.mean(), 1 - y_test.mean()), 3))

0.6
